# Step0_two_leg_open_circuit - Two-Leg Transformer Open-Circuit Test

First transformer model: a square ferrite frame with the primary wound on one leg and the secondary on the opposite leg, solved as an open-circuit test at 1 kHz. It's a quick starting point to see how far the coupling falls short of an ideal transformer before moving on to a shell-type core with concentric windings.

In [ ]:
import pandas as pd
import os
import ansys.aedt.core

In [ ]:
DT = ansys.aedt.core.Desktop(version="2025.2", non_graphical=False, student_version=True)
DT.disable_autosave()
sol_type = "EddyCurrent"
M3D = ansys.aedt.core.maxwell.Maxwell3d(solution_type=sol_type)
oDesign = M3D.odesign

In [ ]:
## Set up output directory for results ##

proj_name = "Step0_two_leg_open_circuit"

# If ANSYS_PROJECT_DIR is set on this machine, save there.
# Otherwise, fall back to the notebook's own working directory,
# so this stays portable for anyone else running the notebook as-is.
base_dir = os.environ.get("ANSYS_PROJECT_DIR", os.getcwd())
dir = os.path.join(base_dir, proj_name)
print(dir)
os.makedirs(dir, exist_ok=True)

desi_name = "Step0_two_leg_open_circuit"

In [ ]:
## Save project and apply design name ##

proj = M3D.oproject
proj.SaveAs(f"{dir}\\{proj_name}.aedt", True)
M3D.rename_design(desi_name, save=False)

# Save again so the design-name change is committed to disk
M3D.save_project()

In [ ]:
## Design parameters ##

core_x = 100         # core depth along x (mm)
core_y = 100         # core outer size along y (mm)
core_z = 100         # core outer size along z (mm)
corehole_x = 100     # window depth, same as core_x so the window goes all the way through
corehole_y = 60      # window size along y (mm)
corehole_z = 60      # window size along z (mm)
coil_turn1 = 20      # primary turns
coil_turn2 = 10      # secondary turns
airgap = 5           # total clearance between a turn and its core leg (2.5 mm on each side)

v_in = 50            # primary source voltage (V)
freq = 1             # frequency (kHz)
input_current_b = 0  # secondary current (A), 0 means open circuit

W_mode = 1
# W_mode = 1 : wire diameter is picked from the larger turn count and the window height
# W_mode = 2 : use coilwidth as the wire diameter (make sure the turns don't overlap)
coilwidth = 5

# Ferrite core material
material_name = "CustomFerrite1"
permittivity = 15
permeability = 1800
mass_density = 4600            # kg/m^3
conductivity = 0.01            # S/m
magnetic_loss_tangent = 0
dielectric_loss_tangent = 0

In [ ]:
## Check core dimensions ##

def check_core_dimensions(core_x, core_y, core_z, corehole_x, corehole_y, corehole_z):
    errors = []

    if core_x != corehole_x:
        errors.append(f"core_x ({core_x}) must equal corehole_x ({corehole_x}) so the window goes through the core.")
    if core_y <= corehole_y:
        errors.append(f"core_y ({core_y}) must be larger than corehole_y ({corehole_y}).")
    if core_z <= corehole_z:
        errors.append(f"core_z ({core_z}) must be larger than corehole_z ({corehole_z}).")
    if (core_y - corehole_y) != (core_z - corehole_z):
        diff_y = core_y - corehole_y
        diff_z = core_z - corehole_z
        errors.append(f"Leg width along y ({diff_y}) and z ({diff_z}) must be the same.")

    if errors:
        for e in errors:
            print(e)
        raise ValueError("Invalid core dimensions.")
    else:
        print("Core dimensions OK.")


check_core_dimensions(core_x, core_y, core_z, corehole_x, corehole_y, corehole_z)

In [ ]:
## Create core material ##

M3D.materials.add_material(
    name=material_name,
    properties={
        "permittivity": permittivity,
        "permeability": permeability,
        "mass_density": mass_density,
        "conductivity": conductivity,
        "magnetic_loss_tangent": magnetic_loss_tangent,
        "dielectric_loss_tangent": dielectric_loss_tangent
    }
)

# Steinmetz core loss coefficients for the ferrite
M3D.materials[material_name].set_power_ferrite_coreloss(
    cm=0.0016,
    x=1.3,
    y=2.1,
    kdc=0,
    cut_depth=0.0001
)

In [ ]:
## Create core ##

# Outer block minus the window gives a square frame with four legs
origin = [-core_x/2, -core_y/2, -core_z/2]
sizes = [core_x, core_y, core_z]
box1 = M3D.modeler.create_box(origin, sizes, name="box1", material=material_name)

origin = [-corehole_x/2, -corehole_y/2, -corehole_z/2]
sizes = [corehole_x, corehole_y, corehole_z]
box2 = M3D.modeler.create_box(origin, sizes, name="box2", material=material_name)

box1.subtract(tool_list=box2, keep_originals=False)

In [ ]:
## Turn path ##

# One rectangular turn around a leg, drawn at the origin and moved into place later
test_points_2 = [[0 ,0 ,0],
                [0 ,(core_y-corehole_y)/2+airgap ,0],
                [core_x+airgap ,(core_y-corehole_y)/2+airgap ,0],
                [core_x+airgap ,0 ,0],
                [0 ,0 ,0]]

In [ ]:
## Create primary and secondary turns ##

# Wire diameter is set by the winding with more turns, so both windings use the same wire
max_turn = max(coil_turn1, coil_turn2)
xsection_width_common = ((corehole_z - 1) / max_turn) * 0.9
if W_mode == 1:
    xsection_width = xsection_width_common
elif W_mode == 2:
    xsection_width = coilwidth
else:
    raise ValueError("Invalid W_mode. Use 1 or 2.")

# Primary turns
for i in range(1, coil_turn1 + 1):
    globals()[f"L{i}"] = M3D.modeler.create_polyline(
        test_points_2,
        segment_type=None,
        cover_surface=False,
        close_surface=False,
        name=f"L{i}",
        material=None,
        xsection_type="Circle",
        xsection_orient=None,
        xsection_width=xsection_width,
        xsection_topwidth=0,
        xsection_height=0,
        xsection_num_seg=6,
        xsection_bend_type=None,
        non_model=False
    )

# Secondary turns
for i in range(1, coil_turn2 + 1):
    globals()[f"R{i}"] = M3D.modeler.create_polyline(
        test_points_2,
        segment_type=None,
        cover_surface=False,
        close_surface=False,
        name=f"R{i}",
        material=None,
        xsection_type="Circle",
        xsection_orient=None,
        xsection_width=xsection_width,
        xsection_topwidth=0,
        xsection_height=0,
        xsection_num_seg=6,
        xsection_bend_type=None,
        non_model=False
    )

In [ ]:
## Assign copper to the turns ##

for i in range(1, coil_turn1 + 1):
    M3D.assign_material(assignment=globals()[f"L{i}"], material="copper")

for i in range(1, coil_turn2 + 1):
    M3D.assign_material(assignment=globals()[f"R{i}"], material="copper")

In [ ]:
## Place primary turns on the +y leg ##

# Turns are spread evenly over the window height, from top to bottom
z_start = corehole_z/2 -((corehole_z)/(coil_turn1)*0.5)
step = ((corehole_z)/(coil_turn1))

for i in range(1, coil_turn1 + 1):
    z_pos = z_start - step * (i - 1)
    M3D.modeler.move(globals()[f"L{i}"], [-(core_x)/2-airgap/2,corehole_y/2-airgap/2, z_pos])

In [ ]:
## Place secondary turns on the -y leg ##

z_start = corehole_z/2 -((corehole_z)/(coil_turn2)*0.5)
step = ((corehole_z)/(coil_turn2))

for i in range(1, coil_turn2 + 1):
    z_pos = z_start - step * (i - 1)
    M3D.modeler.move(globals()[f"R{i}"], [-(core_x)/2-airgap/2,-core_y/2-airgap/2, z_pos])

In [ ]:
## Cut each turn on the YZ plane to get terminal sheets ##

# Each closed turn crosses x = 0 twice, so the section gives two sheets per turn
for i in range(1, coil_turn1 + 1):
    obj = globals()[f"L{i}"]
    section_name = f"L{i}_section"

    M3D.modeler.section(assignment=obj, plane="YZ", create_new=True, section_cross_object=False)
    section_sheet = M3D.modeler.sheet_objects[-1]
    globals()[section_name] = [section_sheet]

    globals()[f"separated_L{i}"] = M3D.modeler.separate_bodies(assignment=globals()[section_name], create_group=False)

for i in range(1, coil_turn2 + 1):
    obj = globals()[f"R{i}"]
    section_name = f"R{i}_section"

    M3D.modeler.section(assignment=obj, plane="YZ", create_new=True, section_cross_object=False)
    section_sheet = M3D.modeler.sheet_objects[-1]
    globals()[section_name] = [section_sheet]

    globals()[f"separated_R{i}"] = M3D.modeler.separate_bodies(assignment=globals()[section_name], create_group=False)

In [ ]:
## Keep one terminal sheet per turn ##

# NOTE: which of the two sheets survives depends on the order AEDT separates them.
# The relative polarity between primary and secondary has not been checked yet.
all_objects = M3D.modeler.object_names

objects_to_delete = [obj for obj in all_objects if "Separate1" in obj]
if objects_to_delete:
    M3D.modeler.delete(assignment=objects_to_delete)

all_objects = M3D.modeler.object_names
section_objects = [obj for obj in all_objects if "Section1" in obj]

In [ ]:
## Assign windings and coil terminals ##

# NOTE: assign_winding / assign_coil / add_winding_coils are not verified yet
# against the AC Magnetic rename in 2025 R2. Switch to the raw API if they fail.
primary_winding = M3D.assign_winding(assignment=None, winding_type="Voltage", is_solid=True, current=0, resistance=0, inductance=0, voltage=v_in, parallel_branches=1, phase=0, name="Primary")
secondary_winding = M3D.assign_winding(assignment=None, winding_type="Current", is_solid=False, current=input_current_b, resistance=0, inductance=0, voltage=0, parallel_branches=1, phase=0, name="Secondary")

terminals = []
for f in section_objects:
    temp = M3D.assign_coil(f, conductors_number=1, polarity="Positive", name=None)
    terminals.append(temp)

# Terminals come out in creation order: primary turns first, then secondary
terminal_names = [terminal.name for terminal in terminals]
primary_terminals = terminal_names[:coil_turn1]
secondary_terminals = terminal_names[-coil_turn2:]

M3D.add_winding_coils(primary_winding.name, primary_terminals)
M3D.add_winding_coils(secondary_winding.name, secondary_terminals)

In [ ]:
## Analysis setup ##

# create_setup() fails for EddyCurrent in 2025 R2 (renamed to AC Magnetic), so the setup is inserted directly
setup_name = "Setup1"
oModule = oDesign.GetModule("AnalysisSetup")
oModule.InsertSetup("EddyCurrent",
    [
        f"NAME:{setup_name}",
        "Enabled:=", True,
        [
            "NAME:MeshLink",
            "ImportMesh:=", False
        ],
        "MaximumPasses:=", 20,
        "MinimumPasses:=", 1,
        "MinimumConvergedPasses:=", 1,
        "PercentRefinement:=", 30,
        "SolveFieldOnly:=", False,
        "PercentError:=", 1,
        "SolveMatrixAtLast:=", True,
        "UseNonLinearIterNum:=", False,
        "UseIterativeSolver:=", False,
        "RelativeResidual:=", 1E-05,
        "ComputeForceDensity:=", False,
        "ComputePowerLoss:=", False,
        "Frequency:=", f"{freq}kHz",
        "HasSweepSetup:=", False,
        "UseHighOrderShapeFunc:=", False,
        "UseMuLink:=", False
    ])

In [ ]:
## Air region ##

[xp, xn, yp, yn, zp, zn] = [100, 100, 100, 100, 100, 100]
region = M3D.modeler.create_air_region(x_pos=xp, x_neg=xn, y_pos=yp, y_neg=yn, z_pos=zp, z_neg=zn)

In [ ]:
## Inductance matrix ##

M3D.assign_matrix(
    assignment=[primary_winding.name, secondary_winding.name],
    matrix_name="Matrix1",
    return_path=None,
    group_sources=None,
    branches=None
)

In [ ]:
## Core loss ##

# set_core_losses() rejects the design because 2025 R2 reports the solver as AC Magnetic,
# so core loss is switched on directly. The second argument is "core loss on field".
oModule = oDesign.GetModule("BoundarySetup")
oModule.SetCoreLoss([box1.name], True)

In [ ]:
## Solve ##

M3D.oeditor.FitAll()
oDesign.Analyze(setup_name)

In [ ]:
## Export terminal results ##

expressions = [
    "InputVoltage(Primary)",
    "InducedVoltage(Primary)",
    "InducedVoltage(Secondary)",
    "FluxLinkage(Primary)",
    "Current(Primary)",
    "CoreLoss",
    "Matrix1.L(Primary, Primary)",
    "Matrix1.CplCoef(Primary, Secondary)"
]

# post.create_report() fails here for the same AC Magnetic reason, so the report is built directly
report_name = "Terminal Data Table"
oModule = oDesign.GetModule("ReportSetup")
oModule.CreateReport(report_name, "EddyCurrent", "Data Table", f"{setup_name} : LastAdaptive", [],
    ["Freq:=", ["All"]],
    ["Y Component:=", expressions])

csv_name = f"{proj_name}_terminal_data.csv"
csv_path = csv_name

oModule.ExportToFile(report_name, os.path.abspath(csv_path))

In [ ]:
data = pd.read_csv(csv_path)
data

In [ ]:
## Save project ##

M3D.save_project()